In [1]:
from workflow import (
    Step, Workflow,
    Fileset, Analysis, Plotting, CustomArtifact,
    RunConfig, ExecutorConfig, FacilityConfig,
    render,
)
import workflow.facilities as facilities
from analysis import get_fileset, run_analysis, plot_results, custom_function_remove_last_file

In [2]:
step_fileset = Step(
    name="Fileset",
    step_type=Fileset,
    builder=get_fileset,
    builder_params={"to_print": "\nTEST:\nparameter testing...\nSUCCESS!\n"},
)

step_custom_filtering = Step(
    name="FilesetFiltering",
    step_type=CustomArtifact,
    builder=custom_function_remove_last_file,
)

step_analysis = Step(
    name="SingleMuonAnalysis",
    step_type=Analysis,
    builder=run_analysis,
)

step_plotting = Step(
    name="PlottingMuonAnalysis",
    step_type=Plotting,
    builder="analysis:plot_results",
)

workflow = Workflow()
workflow.add(step_fileset)
workflow.add(step_analysis, depends_on=[step_fileset])
workflow.add(step_plotting, depends_on=[step_analysis])

# Running on vscode, test
config = RunConfig(
    strategy="by_dataset",
    facility=FacilityConfig(name="coffea-casa"),
    executor_config=ExecutorConfig(executor_type="FuturesExecutor"),
)


config = RunConfig(
    strategy="by_dataset",
    facility=FacilityConfig(name="coffea-casa"),
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)

# Too slow, moving to lxplus — change ONE thing
config = RunConfig(
    strategy="by_dataset",
    facility=FacilityConfig(name="lxplus"),        # ← only this changes
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),  # preserved
)

## `RunConfig` — analysis parameters

Controls *what* to run. These are always **workflow-level** and cannot be overridden per-step.

| Field | Purpose |
|---|---|
| `strategy` | `"by_dataset"` → one chunk per dataset; `None` → all datasets together |
| `percentage` | Share of each dataset's files per chunk (e.g. `25` → 4 chunks per dataset) |
| `datasets` | Restrict to specific dataset names |
| `chunk_fraction` | Process only this fraction of chunks — useful for quick tests |
| `cache_dir` | Where cached outputs live |
| `executor_config` | HOW to run — see `ExecutorConfig` below |
| `facility` | WHERE to run — see `FacilityConfig` below |

In [3]:
step_fileset = Step(
    name="Fileset",
    step_type=Fileset,
    builder=get_fileset,
    builder_params={"to_print": "\nTEST:\nparameter testing...\nSUCCESS!\n"},
)

step_custom_filtering = Step(
    name="FilesetFiltering",
    step_type=CustomArtifact,
    builder=custom_function_remove_last_file,
)

step_analysis = Step(
    name="SingleMuonAnalysis",
    step_type=Analysis,
    builder=run_analysis,
)

step_plotting = Step(
    name="PlottingMuonAnalysis",
    step_type=Plotting,
    builder="analysis:plot_results",
)

workflow = Workflow()
workflow.add(step_fileset)
workflow.add(step_analysis, depends_on=[step_fileset])
workflow.add(step_plotting, depends_on=[step_analysis])

Step(name='PlottingMuonAnalysis', step_type=<class 'workflow.artifacts.Plotting'>, builder='analysis:plot_results', builder_params=None, facility=None, executor_config=None)

## `RunConfig` — analysis parameters

Controls *what* to run. These are always **workflow-wide** and cannot be overridden per-step.

| Field | Purpose |
|---|---|
| `strategy` | `"by_dataset"` → one chunk per dataset; `None` → all datasets together |
| `percentage` | Share of each dataset's files per chunk (e.g. `25` → 4 chunks per dataset) |
| `datasets` | Restrict to specific dataset names |
| `chunk_fraction` | Process only this fraction of chunks — useful for quick tests |
| `cache_dir` | Where cached outputs live |
| `executor_config` | HOW to run — see `ExecutorConfig` below |
| `facility` | WHERE to run — see `FacilityConfig` below |

In [4]:
# Minimal — no splitting, no executor managed
config = RunConfig(cache_dir="cache")

# Process each dataset independently, in 25% chunks (4 chunks per dataset)
config = RunConfig(
    strategy="by_dataset",
    percentage=25,
    cache_dir="cache",
)

# Quick test: restrict to one dataset, process only 30% of chunks
config = RunConfig(
    strategy="by_dataset",
    percentage=50,
    datasets=["SingleMuon"],
    chunk_fraction=0.3,
    cache_dir="cache",
)

## `ExecutorConfig` — HOW to run

Controls which coffea executor the workflow builds and injects into your analysis function as `executor=`.

**Three modes:**

**1. Unmanaged** — omit `executor_config`. The workflow does not inject an executor; your analysis function creates its own.

**2. Managed type** — pick a coffea executor type; the workflow constructs it and passes it to your builder.

**3. Custom instance** — pass any executor object you built yourself; the workflow passes it through unchanged.

In [5]:
# 1. Unmanaged — analysis function owns its executor entirely
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    # executor_config omitted
)

# 2a. FuturesExecutor — multi-process local parallelism
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    executor_config=ExecutorConfig(executor_type="FuturesExecutor", workers=4),
)

# 2b. IterativeExecutor — single-threaded, best for debugging
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    executor_config=ExecutorConfig(executor_type="IterativeExecutor"),
)

# 2c. DaskExecutor with explicit scheduler address
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    executor_config=ExecutorConfig(executor_type="DaskExecutor", dask_scheduler="tcp://host:8786"),
)

# 3. Custom executor instance — you control construction, workflow passes it through
from coffea.processor import FuturesExecutor
my_executor = FuturesExecutor(workers=8)
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    executor_config=ExecutorConfig(executor=my_executor),
)

## `FacilityConfig` — WHERE to run

Describes the computing facility. Combined with `ExecutorConfig` at execution time: the facility provides the Dask scheduler address automatically, so you don't have to repeat it in `ExecutorConfig`.

Use pre-built presets from `workflow.facilities`, or build your own `FacilityConfig` with an explicit address.

**Scheduler address resolution order:** explicit `scheduler_address` → env-var → `None` (raises at run time)

| Preset | Env-var read | Pairs with |
|---|---|---|
| `facilities.local` | — | `FuturesExecutor` |
| `facilities.coffea_casa` | `COFFEA_CASA_SCHEDULER` | `DaskExecutor` |
| `facilities.lxplus` | `LXPLUS_DASK_SCHEDULER` | `DaskExecutor` |

In [6]:
# Local machine — no cluster needed
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    facility=facilities.local,
    executor_config=ExecutorConfig(executor_type="FuturesExecutor", workers=4),
)

# coffea-casa — scheduler address read from COFFEA_CASA_SCHEDULER at execution time
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    facility=facilities.coffea_casa,
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)

# lxplus — scheduler address read from LXPLUS_DASK_SCHEDULER at execution time
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    facility=facilities.lxplus,
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)

# Explicit scheduler address — overrides the env-var
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    facility=FacilityConfig(name="coffea-casa", scheduler_address="tcp://my-host:8786"),
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)

# facility only, no executor_config — executor inferred from facility name
# (local → FuturesExecutor(workers=4), coffea-casa/lxplus → DaskExecutor)
config = RunConfig(
    strategy="by_dataset",
    cache_dir="cache",
    facility=facilities.local,
)